In [ ]:
pip install rdflib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 29.9 MB/s eta 0:00:00


## Создание RDF-графа

In [ ]:
import pandas as pd
from google.colab import drive
from rdflib import Graph, URIRef, Literal, Namespace, BNode
from rdflib.namespace import RDF, RDFS, XSD
from urllib.parse import quote

# 1. Загружаем данные
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/razmetka/Разметка_датасет.csv')

# 2. Создаем граф и пространства имен
g = Graph()
ex = Namespace("http://example.org/animal-allergy#")
mod = Namespace("http://example.org/modality#")
g.bind("ex", ex)
g.bind("mod", mod)

In [ ]:
df

,breed,animal_type,coat_type,undercoat,shedding_level,grooming_needs,size,saliva_production,allergen_level,hypoallergic_label,source_type,text_content,audio_content
0,Poodle,dog,curly,0,low,high,medium,low,low,1,text,The Poodle is one of the most popular hypoalle...,-
1,Bichon Frise,dog,curly,0,low,high,small,low,low,1,audio,-,A Bichon-Thrise is the definition of a low-ene...
2,Maltese,dog,long,0,low,high,small,low,low,1,audio,-,"Well, first off, we have the Maltese Poodlemix..."
3,Golden Retriever,dog,short,1,high,low,large,high,high,0,audio,The Golden Retriever is a beloved family dog k...,"Unfortunately, golden retrievers are not consi..."
4,Corgi,dog,short,1,high,high,medium,low,high,0,audio,-,Many people mistakenly believe that dog hair i...
5,Shi-tzu,dog,long,1,low,high,medium,low,low,1,audio,-,"Coming in next, we have the Brussels Griffon. ..."
6,Yorkshire Terrier,dog,long,0,low,high,small,low,low,1,table,-,-
7,Sphynx,cat,hairless,0,low,high,small,low,low,1,table,-,-
8,Siberian Cat,cat,long,1,medium,medium,medium,medium,medium,1,text,"The Siberian is a large, semi-longhaired cat b...",NaN
9,Persian Cat,cat,long,1,high,high,medium,low,high,0,text,The Persian is one of the oldest and most reco...,-


In [ ]:
# 3. Определяем онтологию
# Классы
g.add((ex.Animal, RDF.type, RDFS.Class))
g.add((ex.Breed, RDF.type, RDFS.Class))
g.add((ex.AllergenicityAssessment, RDF.type, RDFS.Class))
g.add((ex.Modality, RDF.type, RDFS.Class))
g.add((ex.PhysicalCharacteristic, RDF.type, RDFS.Class))

# Свойства для животного
g.add((ex.hasBreed, RDF.type, RDF.Property))
g.add((ex.hasAnimalType, RDF.type, RDF.Property))
g.add((ex.hasCoatType, RDF.type, RDF.Property))
g.add((ex.hasUndercoat, RDF.type, RDF.Property))
g.add((ex.hasSheddingLevel, RDF.type, RDF.Property))
g.add((ex.hasGroomingNeeds, RDF.type, RDF.Property))
g.add((ex.hasSize, RDF.type, RDF.Property))
g.add((ex.hasSalivaProduction, RDF.type, RDF.Property))
g.add((ex.hasAllergenLevel, RDF.type, RDF.Property))

# Свойства для оценки
g.add((ex.isHypoallergenic, RDF.type, RDF.Property))
g.add((ex.fromModality, RDF.type, RDF.Property))
g.add((ex.hasSourceText, RDF.type, RDF.Property))
g.add((ex.hasSourceAudio, RDF.type, RDF.Property))
g.add((ex.hasConfidenceScore, RDF.type, RDF.Property))

<Graph identifier=Nf27a721deff048028521e2f871bd5482 (<class 'rdflib.graph.Graph'>)>

In [ ]:
# 4. Словарь модальностей
modality_uris = {
    "text": URIRef("http://example.org/modality/TextAnalysis"),
    "audio": URIRef("http://example.org/modality/AudioTranscription"),
    "table": URIRef("http://example.org/modality/TabularData")
}

for mod_uri in modality_uris.values():
    g.add((mod_uri, RDF.type, ex.Modality))

In [ ]:
# 5. Заполняем граф данными
for idx, row in df.iterrows():
    breed_name = row['breed']
    # Создаем URI для породы (без пробелов)
    breed_uri = ex[quote(breed_name.replace(' ', '_').replace('-', '_'))]

    # Добавляем информацию о породе
    g.add((breed_uri, RDF.type, ex.Breed))
    g.add((breed_uri, ex.hasName, Literal(breed_name, lang="en")))
    g.add((breed_uri, ex.hasAnimalType, Literal(row['animal_type'], lang="en")))
    g.add((breed_uri, ex.hasCoatType, Literal(row['coat_type'], lang="en")))
    g.add((breed_uri, ex.hasUndercoat, Literal(bool(int(row['undercoat'])), datatype=XSD.boolean)))
    g.add((breed_uri, ex.hasSheddingLevel, Literal(row['shedding_level'], lang="en")))
    g.add((breed_uri, ex.hasGroomingNeeds, Literal(row['grooming_needs'], lang="en")))
    g.add((breed_uri, ex.hasSize, Literal(row['size'], lang="en")))
    g.add((breed_uri, ex.hasSalivaProduction, Literal(row['saliva_production'], lang="en")))
    g.add((breed_uri, ex.hasAllergenLevel, Literal(row['allergen_level'], lang="en")))

    # Создаем животное как экземпляр
    animal_uri = ex[f"Animal_{quote(breed_name.replace(' ', '_').replace('-', '_'))}"]
    g.add((animal_uri, RDF.type, ex.Animal))
    g.add((animal_uri, ex.hasBreed, breed_uri))

    # Добавляем оценку гипоаллергенности с указанием модальности
    if pd.notna(row['hypoallergic_label']):
        assessment_uri = ex[f"Assessment_{quote(breed_name.replace(' ', '_'))}_{row['source_type']}"]
        g.add((assessment_uri, RDF.type, ex.AllergenicityAssessment))
        g.add((assessment_uri, ex.hasAnimal, animal_uri))
        g.add((assessment_uri, ex.isHypoallergenic,
               Literal(bool(int(row['hypoallergic_label'])), datatype=XSD.boolean)))
        g.add((assessment_uri, ex.fromModality, modality_uris[row['source_type']]))

        # Добавляем исходные тексты/аудио если есть
        if pd.notna(row['text_content']) and row['source_type'] == 'text':
            g.add((assessment_uri, ex.hasSourceText, Literal(row['text_content'][:1000], lang="en")))
        if pd.notna(row['audio_content']) and row['source_type'] == 'audio':
            g.add((assessment_uri, ex.hasSourceAudio, Literal(row['audio_content'][:1000], lang="en")))

        # Добавляем уверенность на основе модальности
        confidence_map = {"text": 0.9, "audio": 0.85, "table": 0.75}
        g.add((assessment_uri, ex.hasConfidenceScore,
               Literal(confidence_map[row['source_type']], datatype=XSD.float)))

In [ ]:
# 6. Сохраняем граф
g.serialize(destination="animal_allergy_graph_updated.ttl", format="turtle")
print(f"Граф сохранен в 'animal_allergy_graph_updated.ttl'")
print(f"Всего троек (утверждений): {len(g)}")

Граф сохранен в 'animal_allergy_graph_updated.ttl'
Всего троек (утверждений): 269


Визуализируем:

In [ ]:
# Загружаем граф
g = Graph()
g.parse("animal_allergy_graph_updated.ttl", format="turtle")

def simple_graph_view(graph, max_triples=50):
    count = 0
    for s, p, o in graph:
        if count >= max_triples:
            print(f"\n... и еще {len(graph) - max_triples} связей")
            break

        # Укорачиваем URI для читаемости
        s_str = str(s).replace("http://example.org/animal-allergy#", "ex:")
        s_str = s_str.replace("http://example.org/modality#", "mod:")
        p_str = str(p).replace("http://example.org/animal-allergy#", "ex:")
        o_str = str(o).replace("http://example.org/animal-allergy#", "ex:")
        o_str = o_str.replace("http://example.org/modality#", "mod:")

        # Обрезаем длинные тексты
        if len(o_str) > 50:
            o_str = o_str[:47] + "..."

        print(f"{s_str:40} -> {p_str:25} -> {o_str}")
        count += 1

# Показываем простую версию
simple_graph_view(g, max_triples=30)

# 1. Простая текстовая визуализация
simple_graph_view(g, max_triples=30)

ex:Assessment_Maltese_audio              -> http://www.w3.org/1999/02/22-rdf-syntax-ns#type -> ex:AllergenicityAssessment
ex:Breed                                 -> http://www.w3.org/1999/02/22-rdf-syntax-ns#type -> http://www.w3.org/2000/01/rdf-schema#Class
ex:Animal_Yorkshire_Terrier              -> http://www.w3.org/1999/02/22-rdf-syntax-ns#type -> ex:Animal
ex:Bengal_Cat                            -> ex:hasName                -> Bengal Cat
ex:Corgi                                 -> ex:hasSize                -> medium
ex:Siamese                               -> ex:hasName                -> Siamese
ex:Persian_Cat                           -> ex:hasUndercoat           -> true
ex:Assessment_Yorkshire_Terrier_table    -> ex:fromModality           -> http://example.org/modality/TabularData
ex:Siamese                               -> ex:hasSize                -> medium
ex:Assessment_Bengal_Cat_table           -> ex:fromModality           -> http://example.org/modality/TabularData
ex:Ass

## SPARQL: Запросы к датасету

In [ ]:
from rdflib import Graph
from rdflib.plugins.sparql import prepareQuery

# Загружаем граф
g = Graph()
g.parse("animal_allergy_graph_updated.ttl", format="turtle")

# Префиксы для запросов
PREFIX = """
PREFIX ex: <http://example.org/animal-allergy#>
PREFIX mod: <http://example.org/modality#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
"""

In [ ]:
# ЗАПРОС 1: Все гипоаллергенные животные с указанием модальности
query1 = PREFIX + """
SELECT ?breedName ?isHypoallergenic ?modality ?confidence
WHERE {
  ?animal ex:hasBreed ?breed .
  ?breed ex:hasName ?breedName .
  ?assessment ex:hasAnimal ?animal ;
              ex:isHypoallergenic ?isHypoallergenic ;
              ex:fromModality ?modalityURI ;
              ex:hasConfidenceScore ?confidence .
  BIND(REPLACE(STR(?modalityURI), "http://example.org/modality/", "") AS ?modality)
}
ORDER BY DESC(?confidence)
"""

print("ЗАПРОС 1: Все оценки гипоаллергенности по модальностям")
print("="*60)
for row in g.query(query1):
    hypo = "Гипоаллергенный!" if row.isHypoallergenic else "Не гипоаллергенный!"
    print(f"{row.breedName:20} | {hypo:20} | Модальность: {row.modality:8} | Уверенность: {row.confidence}")


ЗАПРОС 1: Все оценки гипоаллергенности по модальностям
British Shorthair    | Не гипоаллергенный!  | Модальность: TextAnalysis | Уверенность: 0.9
Persian Cat          | Не гипоаллергенный!  | Модальность: TextAnalysis | Уверенность: 0.9
Poodle               | Гипоаллергенный!     | Модальность: TextAnalysis | Уверенность: 0.9
Siberian Cat         | Гипоаллергенный!     | Модальность: TextAnalysis | Уверенность: 0.9
Bichon Frise         | Гипоаллергенный!     | Модальность: AudioTranscription | Уверенность: 0.85
Corgi                | Не гипоаллергенный!  | Модальность: AudioTranscription | Уверенность: 0.85
Golden Retriever     | Не гипоаллергенный!  | Модальность: AudioTranscription | Уверенность: 0.85
Maltese              | Гипоаллергенный!     | Модальность: AudioTranscription | Уверенность: 0.85
Shi-tzu              | Гипоаллергенный!     | Модальность: AudioTranscription | Уверенность: 0.85
Bengal Cat           | Гипоаллергенный!     | Модальность: TabularData | Уверенность: 0.75


In [ ]:
# ЗАПРОС 2: Статистика по модальностям (сколько животных оценено каждой модальностью)
query2 = PREFIX + """
SELECT ?modality (COUNT(DISTINCT ?breedName) AS ?animalCount)
WHERE {
  ?animal ex:hasBreed ?breed .
  ?breed ex:hasName ?breedName .
  ?assessment ex:hasAnimal ?animal ;
              ex:fromModality ?modalityURI .
  BIND(REPLACE(STR(?modalityURI), "http://example.org/modality/", "") AS ?modality)
}
GROUP BY ?modality
ORDER BY DESC(?animalCount)
"""

print("ЗАПРОС 2: Количество животных, оцененных каждой модальностью")
print("="*60)
for row in g.query(query2):
    print(f"{row.modality:10} | {row.animalCount} животных")

ЗАПРОС 2: Количество животных, оцененных каждой модальностью
TabularData | 5 животных
AudioTranscription | 5 животных
TextAnalysis | 4 животных


In [ ]:
# ЗАПРОС 3: Корреляция типа шерсти с гипоаллергенностью
query3 = PREFIX + """
SELECT ?coatType
       (COUNT(?assessment) AS ?total)
       (SUM(IF(?isHypoallergenic = true, 1, 0)) AS ?hypoCount)
WHERE {
  ?animal ex:hasBreed ?breed .
  ?breed ex:hasCoatType ?coatType .
  ?assessment ex:hasAnimal ?animal ;
              ex:isHypoallergenic ?isHypoallergenic .
}
GROUP BY ?coatType
ORDER BY DESC(?hypoCount)
"""

print("ЗАПРОС 3: Анализ гипоаллергенности по типу шерсти")
print("="*60)
for row in g.query(query3):
  percentage = (row.hypoCount.value / row.total.value) * 100 if row.total.value > 0 else 0
  print(f"Тип шерсти: {row.coatType:12} | Гипоаллергенных: {row.hypoCount}/{row.total} ({percentage:.1f}%)")



ЗАПРОС 3: Анализ гипоаллергенности по типу шерсти
Тип шерсти: long         | Гипоаллергенных: 4/6 (66.7%)
Тип шерсти: short        | Гипоаллергенных: 2/5 (40.0%)
Тип шерсти: curly        | Гипоаллергенных: 2/2 (100.0%)
Тип шерсти: hairless     | Гипоаллергенных: 1/1 (100.0%)


In [ ]:
# ЗАПРОС 4: Животные с высоким уровнем аллергена, но признанные гипоаллергенными (противоречие)
query4 = PREFIX + """
SELECT ?breedName ?allergenLevel ?coatType ?sheddingLevel
WHERE {
  ?animal ex:hasBreed ?breed .
  ?breed ex:hasName ?breedName ;
         ex:hasAllergenLevel ?allergenLevel ;
         ex:hasCoatType ?coatType ;
         ex:hasSheddingLevel ?sheddingLevel .
  ?assessment ex:hasAnimal ?animal ;
              ex:isHypoallergenic true .
  FILTER(?allergenLevel IN ("high", "medium"))
}
ORDER BY DESC(?allergenLevel)
"""

print("ЗАПРОС 4: Неожиданно гипоаллергенные (высокий уровень аллергена, но признаны безопасными)")
print("="*60)
for row in g.query(query4):
    print(f"{row.breedName:20} | Уровень аллергена: {row.allergenLevel:6} | Шерсть: {row.coatType:10} | Линька: {row.sheddingLevel}")


ЗАПРОС 4: Неожиданно гипоаллергенные (высокий уровень аллергена, но признаны безопасными)


In [ ]:
# ЗАПРОС 5: Детальная информация по конкретной породе (например, Siberian Cat)
query5 = PREFIX + """
SELECT ?breedName ?animalType ?coatType ?sheddingLevel ?allergenLevel
       ?isHypoallergenic ?modality ?sourceText
WHERE {
  ?breed ex:hasName ?breedName ;
         ex:hasAnimalType ?animalType ;
         ex:hasCoatType ?coatType ;
         ex:hasSheddingLevel ?sheddingLevel ;
         ex:hasAllergenLevel ?allergenLevel .
  FILTER(CONTAINS(LCASE(?breedName), "siberian"))
  ?animal ex:hasBreed ?breed .
  ?assessment ex:hasAnimal ?animal ;
              ex:isHypoallergenic ?isHypoallergenic ;
              ex:fromModality ?modalityURI .
  BIND(REPLACE(STR(?modalityURI), "http://example.org/modality/", "") AS ?modality)
  OPTIONAL { ?assessment ex:hasSourceText ?sourceText }
}
"""

print("ЗАПРОС 5: Детальная информация о Siberian Cat")
print("="*60)
for row in g.query(query5):
    print(f"Порода: {row.breedName}")
    print(f"Тип: {row.animalType}")
    print(f"Шерсть: {row.coatType}, Линька: {row.sheddingLevel}")
    print(f"Уровень аллергена: {row.allergenLevel}")
    print(f"Гипоаллергенный: {'Да' if row.isHypoallergenic else 'Нет'}")
    print(f"Источник оценки: {row.modality}")
    if row.sourceText:
        print(f"Текст: {row.sourceText[:200]}...")
    print("-" * 40)

ЗАПРОС 5: Детальная информация о Siberian Cat
Порода: Siberian Cat
Тип: cat
Шерсть: long, Линька: medium
Уровень аллергена: medium
Гипоаллергенный: Да
Источник оценки: TextAnalysis
Текст: The Siberian is a large, semi-longhaired cat breed originally from Russia. Despite its thick and dense coat, the Siberian is often listed among hypoallergenic cat breeds. Studies suggest that Siberian...
----------------------------------------


In [ ]:
# ЗАПРОС 6: Рейтинг надежности модальностей (средняя уверенность)
query6 = PREFIX + """
SELECT ?modality (AVG(?confidence) AS ?avgConfidence) (COUNT(?assessment) AS ?assessmentsCount)
WHERE {
  ?assessment ex:fromModality ?modalityURI ;
              ex:hasConfidenceScore ?confidence .
  BIND(REPLACE(STR(?modalityURI), "http://example.org/modality/", "") AS ?modality)
}
GROUP BY ?modality
ORDER BY DESC(?avgConfidence)
"""

print("ЗАПРОС 6: Рейтинг модальностей по надежности")
print("="*60)
for row in g.query(query6):
    print(f"{row.modality:10} | Средняя уверенность: {row.avgConfidence.value:.2f} | Оценок: {row.assessmentsCount}")

ЗАПРОС 6: Рейтинг модальностей по надежности
TextAnalysis | Средняя уверенность: 0.90 | Оценок: 4
AudioTranscription | Средняя уверенность: 0.85 | Оценок: 5
TabularData | Средняя уверенность: 0.75 | Оценок: 5


In [ ]:
query7_safe = PREFIX + """
SELECT ?breedName ?animalType ?coatType ?sheddingLevel ?allergenLevel ?modality
WHERE {
  ?animal ex:hasBreed ?breed .
  ?breed ex:hasName ?breedName ;
         ex:hasAnimalType ?animalType ;
         ex:hasCoatType ?coatType ;
         ex:hasSheddingLevel ?sheddingLevel ;
         ex:hasAllergenLevel ?allergenLevel .
  ?assessment ex:hasAnimal ?animal ;
              ex:isHypoallergenic true ;
              ex:fromModality ?modalityURI .
  BIND(REPLACE(STR(?modalityURI), "http://example.org/modality/", "") AS ?modality)
  # Убираем пробелы и приводим к нижнему регистру!!! - без этого не фильтрует нормально
  BIND(LCASE(REPLACE(STR(?allergenLevel), " ", "")) AS ?normalizedLevel)
  FILTER(?normalizedLevel = "low")
}
ORDER BY ?breedName
"""

print("ЗАПРОС 7: Все гипоаллергенные животные с низким уровнем аллергена")
print("="*70)

results = list(g.query(query7_safe))
for row in results:
  print(f"{row.breedName:20} | '{row.allergenLevel}' | {row.modality}")

ЗАПРОС 7: Все гипоаллергенные животные с низким уровнем аллергена
Bengal Cat           | 'low' | TabularData
Bichon Frise         | 'low' | AudioTranscription
Maltese              | 'low' | AudioTranscription
Poodle               | 'low' | TextAnalysis
Shi-tzu              | 'low' | AudioTranscription
Siamese              | 'low' | TabularData
Sphynx               | 'low' | TabularData
Yorkshire Terrier    | 'low' | TabularData
